In [1]:
!pip install -q opencv-python matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q git+https://github.com/huggingface/transformers.git


In [4]:
!pip install torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 2.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 MB 6.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.9 MB/s eta 0:00:00-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 5.5 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully uninstalled sympy-1.13.1
  Attempting uninstall: torch
    Found existing installation: torch 2.5.1
    Uninstalling torch-2.5.1:
      Successfully uninstalled torch-2.5.1


In [7]:
import torch
from segment_anything import sam_model_registry, SamPredictor
import os

# Crea carpeta para el checkpoint
os.makedirs("checkpoints", exist_ok=True)

# Descargar SAM vit_b si no está ya
!wget -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P checkpoints/

# Cargar el modelo
sam_checkpoint = "checkpoints/sam_vit_b_01ec64.pth"
model_type = "vit_b"
device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)

zsh:1: command not found: wget


In [11]:
import os
import cv2
import numpy as np
from segment_anything import sam_model_registry, SamPredictor
import torch

# Configura paths
image_folder = "images"        # carpeta con tus 200 imágenes
output_folder = "masks"        # carpeta de salida
os.makedirs(output_folder, exist_ok=True)

# Modelo
sam_checkpoint = "checkpoints/sam_vit_b_01ec64.pth"
model_type = "vit_b"

# Cargar modelo
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device)
predictor = SamPredictor(sam)

# Procesar imágenes
for filename in os.listdir(image_folder):
    if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    path = os.path.join(image_folder, filename)
    image_bgr = cv2.imread(path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    predictor.set_image(image_rgb)
    masks, _, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        multimask_output=True
    )

    # Elegir la máscara más grande
    areas = [np.sum(mask) for mask in masks]
    largest_idx = np.argmax(areas)
    mask = masks[largest_idx]

    # Guardar máscara binaria
    out_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_mask.png")
    mask_img = (mask * 255).astype(np.uint8)
    cv2.imwrite(out_path, mask_img)

    print(f"[✓] Segmentado: {filename}")


[✓] Segmentado: 0A35GMOFMJRD.jpg
[✓] Segmentado: 8J14I5GAUI1L.jpg
[✓] Segmentado: 7N2JMLCPY41T.jpg
[✓] Segmentado: CF9VAXESOT6H.jpg
[✓] Segmentado: 240W51VC10BB.jpg
[✓] Segmentado: 2072J2QF85Q9.jpg
[✓] Segmentado: 43N1OKMH9L2V.jpg
[✓] Segmentado: 19FGW96UVW4S.jpg
[✓] Segmentado: 0IBBFSH5WBKJ.jpg
[✓] Segmentado: 121B3F1XUICJ.jpg
[✓] Segmentado: BZC74IWR4TUD.jpg
[✓] Segmentado: 3XSLVL4RVMA9.jpg
[✓] Segmentado: 0CRSEPLQ6GCS.jpg
[✓] Segmentado: 66MVI4RA9XRV.jpg
[✓] Segmentado: 29560ASZ5RBX.jpg
[✓] Segmentado: 9UAMHMORSRHZ.jpg
[✓] Segmentado: 0THX9P4F6V3G.jpg
[✓] Segmentado: 7AE8QHZI3ZFT.jpg
[✓] Segmentado: AIX6CSIN52KB.jpg
[✓] Segmentado: 5B4B1RJS33BQ.jpg
[✓] Segmentado: A5TRH9SY8LAQ.jpg
[✓] Segmentado: CCKA02LXCB6N.jpg
[✓] Segmentado: 0X546SO60VWV.jpg
[✓] Segmentado: 3VVZ3M9U8TDG.jpg
[✓] Segmentado: 7074US2Y438C.jpg
[✓] Segmentado: 8F3MXEUXVTFP.jpg
[✓] Segmentado: 8KEBC05TZFUX.jpg
[✓] Segmentado: BUHXSX09H6SA.jpg
[✓] Segmentado: 6NPVR3KMRE7I.jpg
[✓] Segmentado: 0KCZN684A6K9.jpg
[✓] Segmen